# SHL Grammar Scoring Engine

## Objective

Predict the grammar score (0–5) of spoken English audio recordings using acoustic and
automatic speech recognition (ASR) features.

## Dataset

- Training samples: 769
- Test samples: 216
- Target: Grammar score from 0 to 5
- Evaluation metric: RMSE
- Audio: WAV, approximately 45–60 seconds per recording

## Approach

The solution combines two complementary sources of information:

1. **Acoustic features**
   - 127 features extracted using Librosa
   - MFCC statistics
   - Spectral features
   - Zero-crossing rate
   - RMS energy
   - Pitch/F0
   - Speech/pause characteristics
   - Duration

2. **ASR / transcript features**
   - Transcriptions generated using Whisper Small
   - Lexical and structural text features
   - Word and vocabulary statistics
   - Repetition and filler statistics
   - Grammar-marker counts
   - Whisper confidence / speech-detection metadata

This produces **143 model features** in total.

## Model

ExtraTreesRegressor was selected through frozen 5-fold cross-validation.

Final configuration:

- 1000 trees
- `max_features=1.0`
- `min_samples_leaf=2`
- `random_state=42`

## Results

| Evaluation | RMSE |
|---|---:|
| Mean baseline | 1.2383 |
| Acoustic-only ExtraTrees | 0.7500 |
| Combined acoustic + ASR CV | **0.6564** |
| Public Kaggle leaderboard | **0.5131** |

The final submission was generated using the model trained on all 769 training samples.

> **Note:** The training RMSE reported later in the notebook is an in-sample metric and should not be interpreted as an unbiased estimate of generalization performance.

In [70]:
# ============================================================
# 1. Imports, paths and configuration
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

warnings.filterwarnings("ignore")

# Competition paths
BASE = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
CACHE = "/kaggle/working/cache"

TRAIN_CSV = os.path.join(BASE, "train.csv")
TEST_CSV = os.path.join(BASE, "test.csv")

ACOUSTIC_CACHE = os.path.join(CACHE, "acoustic.csv")
TRANSCRIPT_CACHE = os.path.join(CACHE, "transcripts_v3_small.jsonl")
TRANSCRIPT_FEATURES = os.path.join(CACHE, "transcript_features_v1.csv")
COMBINED_FEATURES = os.path.join(CACHE, "combined_features_v1.csv")
FOLDS_FILE = os.path.join(CACHE, "folds_5fold_seed42_v1.csv")

SUBMISSION_PATH = "/kaggle/working/submission_test216.csv"

RANDOM_STATE = 42

print("Configuration loaded.")
print("Dataset:", BASE)
print("Cache:", CACHE)
print("Random state:", RANDOM_STATE)

Configuration loaded.
Dataset: /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final
Cache: /kaggle/working/cache
Random state: 42


In [71]:
# ============================================================
# 2. Load competition data
# ============================================================

train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

# Create collision-safe keys.
# The same filename can occur in both splits, so split/filename
# is the unique identifier.
train["key"] = "train/" + train["filename"]
test["key"] = "test/" + test["filename"]

print("Train shape:", train.shape)
print("Test shape :", test.shape)

print("\nTrain target summary:")
print(train["label"].describe())

# Basic integrity checks
assert len(train) == 769
assert len(test) == 216
assert train["filename"].notna().all()
assert test["filename"].notna().all()
assert train["label"].notna().all()
assert train["key"].is_unique
assert test["key"].is_unique

print("\nTarget distribution:")
print(train["label"].value_counts().sort_index())

print("\nData integrity checks: PASSED")

Train shape: (769, 3)
Test shape : (216, 3)

Train target summary:
count    769.000000
mean       3.313394
std        1.239000
min        0.000000
25%        2.500000
50%        3.000000
75%        4.000000
max        5.000000
Name: label, dtype: float64

Target distribution:
label
0.0     37
1.0      1
1.5      3
2.0    102
2.5     79
3.0    174
3.5     64
4.0    124
4.5     52
5.0    133
Name: count, dtype: int64

Data integrity checks: PASSED


In [72]:
# ============================================================
# 3. Load validated combined feature matrix
# ============================================================

combined = pd.read_csv(COMBINED_FEATURES)

print("Combined feature matrix:", combined.shape)

# Expected structure:
# 985 rows = 769 train + 216 test
# 143 model features + 1 key column

assert combined.shape == (985, 144)
assert combined["key"].is_unique
assert combined.isna().sum().sum() == 0

# Verify exact train/test keys
expected_keys = list(train["key"]) + list(test["key"])

assert set(combined["key"]) == set(expected_keys)

# Align feature matrix to the exact dataset order
combined_indexed = combined.set_index("key")

X_train = combined_indexed.loc[train["key"]].copy()
X_test = combined_indexed.loc[test["key"]].copy()

y_train = train["label"].astype(float).values

assert X_train.shape == (769, 143)
assert X_test.shape == (216, 143)

print("Train feature matrix:", X_train.shape)
print("Test feature matrix :", X_test.shape)
print("Missing values      :", X_train.isna().sum().sum() + X_test.isna().sum().sum())
print("\nFeature matrix checks: PASSED")

Combined feature matrix: (985, 144)
Train feature matrix: (769, 143)
Test feature matrix : (216, 143)
Missing values      : 0

Feature matrix checks: PASSED


In [74]:
# ============================================================
# 4. Frozen 5-fold cross-validation setup
# ============================================================

# Stratification bins chosen to preserve the important target ranges
def make_strat_bins(y):
    return np.select(
        [
            y == 0,
            (y > 0) & (y < 2.5),
            (y >= 2.5) & (y < 3.5),
            (y >= 3.5) & (y < 4.5),
            y >= 4.5
        ],
        [0, 1, 2, 3, 4]
    )

strat_bins = make_strat_bins(y_train)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

folds = np.full(len(train), -1, dtype=int)

for fold, (_, val_idx) in enumerate(skf.split(X_train, strat_bins)):
    folds[val_idx] = fold

assert np.all(folds >= 0)
assert np.bincount(folds).tolist() == [154, 154, 154, 154, 153]

fold_df = pd.DataFrame({
    "key": train["key"],
    "fold": folds
})

print("Fold sizes:")
print(fold_df["fold"].value_counts().sort_index())

print("\nZero-label samples per fold:")
print(
    train.assign(fold=folds)
    .groupby("fold")["label"]
    .apply(lambda x: int((x == 0).sum()))
)

# Save the frozen fold assignment
fold_df.to_csv(FOLDS_FILE, index=False)

print("\nFold file:", FOLDS_FILE)
print("Frozen 5-fold CV setup: PASSED")

Fold sizes:
fold
0    154
1    154
2    154
3    154
4    153
Name: count, dtype: int64

Zero-label samples per fold:
fold
0    7
1    7
2    8
3    8
4    7
Name: label, dtype: int64

Fold file: /kaggle/working/cache/folds_5fold_seed42_v1.csv
Frozen 5-fold CV setup: PASSED


In [75]:
# ============================================================
# 5. Final model: 5-fold cross-validation
# ============================================================

oof_pred = np.zeros(len(X_train))
fold_rmse = []
fold_mae = []

for fold in range(5):
    train_idx = np.where(folds != fold)[0]
    val_idx = np.where(folds == fold)[0]

    model = ExtraTreesRegressor(
        n_estimators=1000,
        max_features=1.0,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(
        X_train.iloc[train_idx],
        y_train[train_idx]
    )

    pred = model.predict(X_train.iloc[val_idx])
    pred = np.clip(pred, 0, 5)

    oof_pred[val_idx] = pred

    rmse = mean_squared_error(
        y_train[val_idx],
        pred
    ) ** 0.5

    mae = mean_absolute_error(
        y_train[val_idx],
        pred
    )

    fold_rmse.append(rmse)
    fold_mae.append(mae)

    print(
        f"Fold {fold}: "
        f"RMSE={rmse:.4f}, "
        f"MAE={mae:.4f}"
    )

pooled_rmse = mean_squared_error(y_train, oof_pred) ** 0.5
pooled_mae = mean_absolute_error(y_train, oof_pred)
pooled_r2 = r2_score(y_train, oof_pred)

print("\n" + "=" * 55)
print("FINAL 5-FOLD CV RESULTS")
print("=" * 55)

print(f"Pooled RMSE : {pooled_rmse:.6f}")
print(f"Mean Fold RMSE: {np.mean(fold_rmse):.6f}")
print(f"Fold RMSE Std : {np.std(fold_rmse):.6f}")
print(f"MAE          : {pooled_mae:.6f}")
print(f"R²           : {pooled_r2:.6f}")

print("\nOOF predictions generated:", len(oof_pred))

Fold 0: RMSE=0.7076, MAE=0.5341
Fold 1: RMSE=0.6724, MAE=0.5319
Fold 2: RMSE=0.6095, MAE=0.4697
Fold 3: RMSE=0.6645, MAE=0.5355
Fold 4: RMSE=0.6229, MAE=0.4911

FINAL 5-FOLD CV RESULTS
Pooled RMSE : 0.656376
Mean Fold RMSE: 0.655379
Fold RMSE Std : 0.035373
MAE          : 0.512478
R²           : 0.718986

OOF predictions generated: 769


In [76]:
# ============================================================
# 6. Train final model on all training data
# ============================================================

final_model = ExtraTreesRegressor(
    n_estimators=1000,
    max_features=1.0,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

# Training predictions (reported only as an in-sample diagnostic)
train_pred = np.clip(
    final_model.predict(X_train),
    0,
    5
)

# Test predictions
test_pred = np.clip(
    final_model.predict(X_test),
    0,
    5
)

train_rmse = mean_squared_error(
    y_train,
    train_pred
) ** 0.5

train_mae = mean_absolute_error(
    y_train,
    train_pred
)

train_r2 = r2_score(
    y_train,
    train_pred
)

print("=" * 55)
print("FINAL MODEL TRAINED")
print("=" * 55)

print(f"Training samples : {len(X_train)}")
print(f"Features         : {X_train.shape[1]}")
print(f"Trees            : 1000")
print(f"Min samples leaf : 2")

print("\nIn-sample training metrics:")
print(f"RMSE : {train_rmse:.6f}")
print(f"MAE  : {train_mae:.6f}")
print(f"R²   : {train_r2:.6f}")

print("\nTest predictions:")
print(f"Count : {len(test_pred)}")
print(f"Min   : {test_pred.min():.4f}")
print(f"Max   : {test_pred.max():.4f}")
print(f"Mean  : {test_pred.mean():.4f}")
print(f"Std   : {test_pred.std():.4f}")

FINAL MODEL TRAINED
Training samples : 769
Features         : 143
Trees            : 1000
Min samples leaf : 2

In-sample training metrics:
RMSE : 0.064608
MAE  : 0.041306
R²   : 0.997277

Test predictions:
Count : 216
Min   : 2.2616
Max   : 4.8231
Mean  : 3.3164
Std   : 0.5562


In [78]:
# ============================================================
# 7. Model interpretability: feature importance
# ============================================================

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": final_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("Top 20 features:")
display(feature_importance.head(20))

# Aggregate acoustic vs transcript importance
acoustic_importance = feature_importance[
    ~feature_importance["feature"].isin([
        "text_chars",
        "word_count",
        "unique_words",
        "lexical_diversity",
        "sentence_count",
        "avg_words_per_sentence",
        "repeated_adjacent",
        "repetition_ratio",
        "filler_count",
        "filler_ratio",
        "long_word_count",
        "long_word_ratio",
        "grammar_marker_count",
        "asr_n_segments",
        "mean_logprob",
        "mean_no_speech"
    ])
]["importance"].sum()

asr_importance = feature_importance[
    feature_importance["feature"].isin([
        "text_chars",
        "word_count",
        "unique_words",
        "lexical_diversity",
        "sentence_count",
        "avg_words_per_sentence",
        "repeated_adjacent",
        "repetition_ratio",
        "filler_count",
        "filler_ratio",
        "long_word_count",
        "long_word_ratio",
        "grammar_marker_count",
        "asr_n_segments",
        "mean_logprob",
        "mean_no_speech"
    ])
]["importance"].sum()

print("\nFeature group importance:")
print(f"Acoustic features    : {acoustic_importance:.4f}")
print(f"Transcript/ASR       : {asr_importance:.4f}")
print(f"Total                : {acoustic_importance + asr_importance:.4f}")

assert abs(
    acoustic_importance + asr_importance - 1.0
) < 1e-6

print("\nInterpretability checks: PASSED")

Top 20 features:


,feature,importance
0,unique_words,0.091641
1,mean_logprob,0.064463
2,zcr_p10,0.064453
3,mean_no_speech,0.049124
4,centroid_p10,0.038015
5,rolloff_p10,0.032942
6,zcr_mean,0.027629
7,dmfcc1_std,0.023571
8,mfcc1_std,0.020586
9,zcr_med,0.020050



Feature group importance:
Acoustic features    : 0.7193
Transcript/ASR       : 0.2807
Total                : 1.0000

Interpretability checks: PASSED


In [79]:
# ============================================================
# 8. Create and validate final Kaggle submission
# ============================================================

submission = pd.DataFrame({
    "filename": test["filename"].values,
    "label": test_pred
})

# Keep exactly the test.csv order
assert len(submission) == len(test)
assert submission["filename"].equals(test["filename"])
assert submission["filename"].is_unique
assert submission["label"].notna().all()
assert submission["label"].between(0, 5).all()

submission.to_csv(SUBMISSION_PATH, index=False)

print("=" * 55)
print("SUBMISSION CREATED")
print("=" * 55)

print("Path:", SUBMISSION_PATH)
print("Rows:", len(submission))
print("Columns:", list(submission.columns))

print("\nPrediction summary:")
print(submission["label"].describe())

print("\nFirst 10 predictions:")
display(submission.head(10))

print("\nValidation:")
print("✓ 216 test rows")
print("✓ Unique filenames")
print("✓ No missing predictions")
print("✓ Predictions constrained to [0, 5]")
print("✓ Test order preserved")

print("\nSubmission checks: PASSED")

SUBMISSION CREATED
Path: /kaggle/working/submission_test216.csv
Rows: 216
Columns: ['filename', 'label']

Prediction summary:
count    216.000000
mean       3.316384
std        0.557539
min        2.261583
25%        2.945187
50%        3.166583
75%        3.670625
max        4.823083
Name: label, dtype: float64

First 10 predictions:


,filename,label
0,audio_128.wav,3.019083
1,audio_156.wav,3.680500
2,audio_19.wav,3.973667
3,audio_108.wav,2.332250
4,audio_206.wav,3.172333
5,audio_161.wav,3.077417
6,audio_215.wav,3.213333
7,audio_213.wav,2.615833
8,audio_11.wav,3.034750
9,audio_155.wav,3.044000



Validation:
✓ 216 test rows
✓ Unique filenames
✓ No missing predictions
✓ Predictions constrained to [0, 5]
✓ Test order preserved

Submission checks: PASSED


# Final Report — SHL Grammar Scoring Engine

## 1. Objective

The task is to predict a grammar score from spoken English audio recordings.

- Training samples: **769**
- Test samples: **216**
- Target range: **0–5**
- Evaluation metric: **RMSE**
- No external training data was used.

## 2. Feature Engineering

The solution combines two complementary sources of information.

### Acoustic features

Audio recordings were processed using `librosa` to extract **127 acoustic features**, including:

- Duration
- RMS energy statistics
- Zero-crossing rate
- Spectral centroid
- Spectral bandwidth
- Spectral rolloff
- Spectral contrast
- MFCCs
- MFCC delta features
- Fundamental frequency (F0)
- Pause and speech-related statistics
- Onset and energy-related features

### ASR / Transcript features

Each recording was transcribed using **Whisper small**.

The transcript and ASR metadata were converted into **16 features**, including:

- Character count
- Word count
- Unique-word count
- Lexical diversity
- Sentence count
- Average words per sentence
- Repetition measures
- Filler-word statistics
- Long-word statistics
- Grammar-marker count
- Number of ASR segments
- Mean log probability
- Mean no-speech probability

This produced a final feature matrix containing:

**143 features per audio sample**

## 3. Model

The final model is an `ExtraTreesRegressor` with:

- Number of trees: **1000**
- `max_features = 1.0`
- `min_samples_leaf = 2`
- `random_state = 42`
- `n_jobs = -1`

Predictions were clipped to the valid grammar-score range of **0–5**.

## 4. Cross-Validation

A frozen **5-fold StratifiedKFold** split was used with:

- `shuffle=True`
- `random_state=42`

The target was stratified into five score ranges to preserve the distribution of very low, low-to-mid, mid, high, and very high grammar scores.

### Cross-validation results

| Metric | Result |
|---|---:|
| Pooled RMSE | **0.656376** |
| Mean Fold RMSE | **0.655379** |
| Fold RMSE Std | **0.035373** |
| MAE | **0.512478** |
| R² | **0.718986** |

Fold RMSE values:

- Fold 0: 0.7076
- Fold 1: 0.6724
- Fold 2: 0.6095
- Fold 3: 0.6645
- Fold 4: 0.6229

## 5. Feature Importance

The final model's feature importance indicates that both acoustic and transcript information contribute to prediction.

| Feature group | Importance |
|---|---:|
| Acoustic features | **71.93%** |
| Transcript / ASR features | **28.07%** |

Top individual features included:

1. `unique_words`
2. `mean_logprob`
3. `zcr_p10`
4. `mean_no_speech`
5. `centroid_p10`
6. `rolloff_p10`
7. `zcr_mean`
8. `dmfcc1_std`
9. `mfcc1_std`
10. `zcr_med`

## 6. Final Training

The final model was trained on all **769 training samples**.

The in-sample training metrics were:

- RMSE: **0.064608**
- MAE: **0.041306**
- R²: **0.997277**

These are **in-sample diagnostics** and should not be interpreted as an unbiased estimate of generalization performance. The 5-fold CV result is the appropriate internal validation estimate.

## 7. Test Prediction

The final model generated predictions for all **216 test recordings**.

Prediction range:

- Minimum: **2.2616**
- Maximum: **4.8231**
- Mean: **3.3164**
- Standard deviation: **0.5575**

The final submission contains:

- 216 rows
- `filename`
- `label`
- No missing predictions
- Unique filenames
- Predictions constrained to [0, 5]
- Original test-file order preserved

## 8. Competition Result

The submitted predictions achieved:

### **Public Kaggle RMSE: 0.5131**

This is the final external evaluation result on the competition leaderboard.

The result improved substantially over the simple mean-target baseline and over the acoustic-only model.

## 9. Final Pipeline

```text
Spoken Audio
     │
     ├──► Acoustic Feature Extraction
     │        └── 127 features
     │
     └──► Whisper Small ASR
              └── Transcript + ASR metadata
                       └── 16 features
     
              ↓
     
        143 Combined Features
              ↓
       ExtraTrees Regressor
       1000 trees, leaf=2
              ↓
        Grammar Score [0,5]
              ↓
        Kaggle Submission